# Data Loading

Load the data into a DataFrame

In [ ]:
import pandas as pd

df = pd.read_csv('Bank Customer Churn Prediction.csv')
display(df.head())

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


# **Part A**

# Part A Question 1
What is the overall churn rate in the dataset?



Calculate the mean of the 'churn' column to find the churn rate.

In [ ]:
churn_rate = df['churn'].mean()*100

print(f"The overall churn rate in the dataset is: {churn_rate:.2f}%")

The overall churn rate in the dataset is: 20.37%


# Part A Question 2
 Provide the distribution of customers by country.




# Part A Question 3
Compare the average credit score between churned and non-churned customers.



In [ ]:
credit_score_comparison = df.groupby('churn')['credit_score'].mean()

print("Average credit score for churned (1) and non-churned (0) customers:")
print(credit_score_comparison)

credit_score_difference = abs(credit_score_comparison[0] - credit_score_comparison[1])

if credit_score_comparison[0] > credit_score_comparison[1]:
    print(f"\nNon-churned customers have a higher average credit score than churned customers by {credit_score_difference:.2f}.")
else:
    print(f"\nChurned customers have a higher average credit score than non-churned customers by {credit_score_difference:.2f}.")

Average credit score for churned (1) and non-churned (0) customers:
churn
0    651.853196
1    645.351497
Name: credit_score, dtype: float64

Non-churned customers have a higher average credit score than churned customers by 6.50.


# **Part B**


# Part B Question 4
Preprocess the dataset appropriately (e.g., categorical encoding, scaling).

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'churn' in numerical_cols:
    numerical_cols.remove('churn')

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['country', 'gender']
Numerical columns: ['customer_id', 'credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (8000, 14)
Shape of X_test: (2000, 14)
Shape of y_train: (8000,)
Shape of y_test: (2000,)


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

X = df.drop('churn', axis=1)
y = df['churn']

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)

print("Shape of processed features:", X_processed.shape)

Shape of processed features: (10000, 14)


## Part B Question 5


Define at least two different classification models (e.g., Logistic Regression, Random Forest, Gradient Boosting).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

print("Initialized models:")
for name, model in models.items():
    print(f"- {name}")

Initialized models:
- Logistic Regression
- Random Forest
- Gradient Boosting


In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)

## Evaluate models


Evaluate each trained model using Accuracy, Precision, Recall, F1-score, and ROC-AUC on the testing data.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

results = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC-AUC': roc_auc
    }

results_df = pd.DataFrame(results).T
display(results_df)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Logistic Regression,0.8085,0.592308,0.189189,0.286778,0.774530
Random Forest,0.8665,0.799145,0.459459,0.583463,0.847632
Gradient Boosting,0.8720,0.793774,0.501229,0.614458,0.870576


## Part B Question 6


Compare the performance metrics of the models to determine which one performs better.



Write a brief comparison of the models based on the performance metrics displayed in the DataFrame, highlighting the best performing model for each metric and overall.



In [ ]:
print("Model Comparison:\n")
print(results_df)

print("\nComparison Summary:")

# Dynamically find the best model for each metric
best_accuracy_model = results_df['Accuracy'].idxmax()
best_precision_model = results_df['Precision'].idxmax()
best_recall_model = results_df['Recall'].idxmax()
best_f1_model = results_df['F1-score'].idxmax()
best_roc_auc_model = results_df['ROC-AUC'].idxmax()


print(f"Accuracy: {best_accuracy_model} ({results_df.loc[best_accuracy_model, 'Accuracy']:.4f}) is the most accurate model.")
print(f"Precision: {best_precision_model} ({results_df.loc[best_precision_model, 'Precision']:.4f}) has the highest precision, meaning fewer false positives.")
print(f"Recall: {best_recall_model} ({results_df.loc[best_recall_model, 'Recall']:.4f}) has the highest recall, meaning it identifies more actual churn cases.")
print(f"F1-score: {best_f1_model} ({results_df.loc[best_f1_model, 'F1-score']:.4f}) has the best balance between precision and recall.")


Model Comparison:

                     Accuracy  Precision    Recall  F1-score   ROC-AUC
Logistic Regression    0.8085   0.592308  0.189189  0.286778  0.774530
Random Forest          0.8665   0.799145  0.459459  0.583463  0.847632
Gradient Boosting      0.8720   0.793774  0.501229  0.614458  0.870576

Comparison Summary:
Accuracy: Gradient Boosting (0.8720) is the most accurate model.
Precision: Random Forest (0.7991) has the highest precision, meaning fewer false positives.
Recall: Gradient Boosting (0.5012) has the highest recall, meaning it identifies more actual churn cases.
F1-score: Gradient Boosting (0.6145) has the best balance between precision and recall.


# Part B Question 7

Among the evaluated models (Logistic Regression, Random Forest, and Gradient Boosting), the Gradient Boosting model demonstrated the best overall performance, achieving the highest Accuracy (0.8720), Recall (0.5012), F1-score (0.6145), and ROC-AUC (0.8706).
The Random Forest model had the highest Precision (0.7991).
Logistic Regression performed significantly worse than Random Forest and Gradient Boosting across all metrics.

#  Part C
PArt C Question 8
Perform feature importance analysis using Gradient Boosting model


In [ ]:
best_model = models['Gradient Boosting']
feature_importances = best_model.feature_importances_


numerical_features = numerical_cols
categorical_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_features = np.concatenate([numerical_features, categorical_features])

feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': feature_importances
})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

display(feature_importance_df)

,Feature,Importance
2,age,0.386774
5,products_number,0.297763
7,active_member,0.112593
4,balance,0.080666
10,country_Germany,0.055906
1,credit_score,0.017900
8,estimated_salary,0.014826
0,customer_id,0.013814
13,gender_Male,0.007538
12,gender_Female,0.005709


## Part C Question 9
Define age group categories and create a new column in the DataFrame for these groups.


In [ ]:
age_bins = [0, 30, 40, 50, 60, df['age'].max()]
age_labels = ['<30', '30-40', '40-50', '50-60', '60+']

df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=True)

print("Distribution of customers by age group:")
display(df['age_group'].value_counts().sort_index())

Distribution of customers by age group:


,count
age_group,
<30,1968
30-40,4451
40-50,2320
50-60,797
60+,464


## Churn rate by age group


In [ ]:
churn_rate_by_age_group = df.groupby('age_group')['churn'].mean()

print("Churn rate by age group:")
display(churn_rate_by_age_group)

Churn rate by age group:


/tmp/ipython-input-560164255.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  churn_rate_by_age_group = df.groupby('age_group')['churn'].mean()


,churn
age_group,
<30,0.075203
30-40,0.120872
40-50,0.339655
50-60,0.562108
60+,0.247845


## Identify high-risk age group



In [ ]:
highest_churn_age_group = churn_rate_by_age_group.idxmax()
highest_churn_rate_value = churn_rate_by_age_group.max()

print(f"The age group with the highest churn rate is: {highest_churn_age_group} with a churn rate of {highest_churn_rate_value:.2f}%.")

The age group with the highest churn rate is: 50-60 with a churn rate of 0.56%.


**Reasoning**:
Provide a brief explanation for the findings in a markdown cell.



The age group 50-60 has the highest churn rate in this dataset. Several factors could contribute to this:

*   **Changing Financial Needs:** Customers in this age range might be approaching retirement, leading to changes in their financial needs and priorities.
*   **Life Events:** Major life events like retirement, children leaving home, or health changes can trigger a review of financial relationships and lead to switching banks.

*   **Targeted Offers from Competitors:** Banks might actively target customers in this age bracket with attractive offers for retirement accounts, wealth management services, or higher interest rates on savings, enticing them to switch.

It's important to note that without further data on customer behavior and preferences within this age group, these are potential explanations based on general demographic trends.

# Part D: Model Validation & Interpretation

## Part D Question 10

Suppose the bank is more concerned about identifying churned customers rather than keeping accuracy high. Which evaluation metric would be the most important in this context? Explain your reasoning.


In this scenario, where the bank is more concerned with identifying churned customers, the most important evaluation metric would be Recall.

*   **Recall** measures the ability of the model to correctly identify all actual churned customers (True Positives).

If the bank's primary goal is to identify as many churned customers as possible, missing a churned customer (a False Negative, which Recall helps minimize) is more costly than incorrectly identifying a non-churned customer as churning (a False Positive, which Precision helps minimize). The bank can tolerate some False Positives and investigate those customers, but they cannot intervene with churned customers they failed to identify. Therefore, maximizing Recall is very important

## Part D Question 11

Based on your models, what are two key factors that strongly influence customer churn?


Based on the feature importance analysis performed using the Gradient Boosting model (which showed the best overall performance), two key factors that strongly influence customer churn are:

1.  **Age:** The feature importance analysis showed that `age` has the highest importance score. This is further supported by the analysis of churn rate by age group, which clearly demonstrated that customers in the 50-60 age group have a significantly higher churn rate compared to other age groups. This suggests that age, particularly being in the older age brackets, is a strong indicator of a customer's likelihood to churn.

2.  **Number of Products:** The `products_number` also has a high feature importance score. This suggests that the number of banking products a customer has is a significant factor in determining churn. Customers with fewer products might be less engaged with the bank and more likely to churn, while those with multiple products might be more integrated and less likely to leave. (Further analysis could explore the specific relationship, e.g., is it a lower number of products or a very high number that is more associated with churn?).

These two factors, age and the number of products, appear to be the most influential in predicting customer churn based on the Gradient Boosting model's feature importance.